# Accumulator-to-spike readout benchmark

One locked cross-user split; model seeds 11/23/37 measure optimization variability, not uncertainty across user splits. R0/R1/R2 reuse the O0 checkpoint; R3 is a separate full end-to-end LIF training run with O0-paired initialization. R4 is a no-retraining accumulator swap on the selected R3 checkpoint. No training or probe fitting is performed by this notebook.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core_benchmark_v1/protocol.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside the WritingRing repository")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from core_benchmark_v1.analysis import results_root, load_manifest, table, plot_native, plot_probe, plot_readout
ROOT = results_root(REPO)
MANIFEST = load_manifest(ROOT)
display(MANIFEST)


## Same checkpoint and same W

R0 is the accumulator. R1_sameW_fixed changes only output dynamics, using theta=0.5. R1_sameW_calibrated uses validation-only threshold calibration. IF beta=1 and LIF beta=0.5 have no output synaptic filter.

In [ ]:
readout = table(ROOT, "readout_summary", "04_readout")
display(readout)
plot_readout(readout)

## Output W adaptation

R2 freezes all hidden representations and adapts only W using CE of valid output spike counts. The threshold is held fixed after validation calibration. A remaining gap is not a proven fundamental limitation.

In [ ]:
display(table(ROOT, "readout_gains", "04_readout"))
display(table(ROOT, "readout_gains_summary", "04_readout"))

## Failure diagnostics

All-zero outputs and tied maximum counts are reported explicitly. Argmax ties use the first class index consistently.

In [ ]:
display(table(ROOT, "readout_runs", "04_readout"))

## Full end-to-end LIF control

`R3_e2e_LIF` trains L1, L2 and W end-to-end through the beta=0.5 output LIF, with the same 234->234 architecture, paired initialization, WCCE valid-mean reduction, optimizer, budget, threshold and bias-free head as O0. `R4_e2e_accumulator_swap` reuses that selected checkpoint and bypasses only the final LIF, accumulating the exact same analog `Wz[t]` evidence. Thus R4-R3 measures residual spike-realization loss after full-network adaptation.

In [ ]:
e2e = readout[readout['case'].isin(['R_LIF', 'R_LIF_E2E'])]
display(e2e)
gains = table(ROOT, 'readout_gains_summary', '04_readout')
display(gains[gains['case'].isin(['R_LIF', 'R_LIF_E2E'])])